# Audit initial du datset "Global Electronics Retailer"

Ce notebook a pour objectif d'explorer et d'évaluer la qualité des données brutes du dataset **Global Electronics Retailer** avant leur exploitation dans PostgreSQL et Power BI. L'audit permettra notamment de vérifier la structure des fichiers, les types de données, les valeurs manquantes, les doublons, l'unicité des clés, l'intégrité des relations entre les tables ainsi que certaines règles de cohérence métier.

## 1. Audit de la table `Sales`

Nous commençons par la table `Sales`, qui constitue la table transactionnelle centrale du dataset, afin de vérifier que les données sont correctement chargées et correspondent à la structure décrite dans le dictionnaire de données.

### 1.1 Chargement et 1ère inspection des données

In [1]:
# Import de la bibliothèque pandas
import pandas as pd

In [2]:
# Chargement des données de ventes dans un DataFrame
sales = pd.read_csv("../data/raw/Sales.csv")

In [3]:
# Aperçu des premières lignes de la table Sales
sales.head()

,Order Number,Line Item,Order Date,Delivery Date,CustomerKey,StoreKey,ProductKey,Quantity,Currency Code
0,366000,1,1/1/2016,NaN,265598,10,1304,1,CAD
1,366001,1,1/1/2016,1/13/2016,1269051,0,1048,2,USD
2,366001,2,1/1/2016,1/13/2016,1269051,0,2007,1,USD
3,366002,1,1/1/2016,1/12/2016,266019,0,1106,7,CAD
4,366002,2,1/1/2016,1/12/2016,266019,0,373,1,CAD


<div style=" display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;
">

<strong>Observation :</strong> Le fichier <code>Sales.csv</code> est correctement chargé et contient les 9 variables attendues d'après le dictionnaire de données. Les premières lignes confirment également qu'une même commande peut comporter plusieurs lignes de produits.

</div>

### 1.2 Vérification de la structure et des dimensions

Cette étape permet de vérifier la structure réelle de la table `Sales` et d'en mesurer le volume. Nous contrôlons le nombre de lignes et de colonnes ainsi que les types de données détectés par Pandas afin d'identifier d'éventuelles incohérences avant de poursuivre l'audit.

In [4]:
# Vérification des dimensions de la table Sales
print(f"Nombre de lignes : {sales.shape[0]}")
print(f"Nombre de colonnes : {sales.shape[1]}")

Nombre de lignes : 62884
Nombre de colonnes : 9


In [5]:
# Vérification des colonnes et de leurs types
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 62884 entries, 0 to 62883
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Order Number   62884 non-null  int64
 1   Line Item      62884 non-null  int64
 2   Order Date     62884 non-null  str  
 3   Delivery Date  13165 non-null  str  
 4   CustomerKey    62884 non-null  int64
 5   StoreKey       62884 non-null  int64
 6   ProductKey     62884 non-null  int64
 7   Quantity       62884 non-null  int64
 8   Currency Code  62884 non-null  str  
dtypes: int64(6), str(3)
memory usage: 4.3 MB


<div style=" display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;
">

<strong>Observation :</strong> La table <code>Sales</code> contient 62 884 lignes de commande et 9 colonnes. Toutes les variables sont complètes à l'exception de `Delivery Date`, renseignée sur seulement 13 165 lignes. <code>Order Date</code> et <code>Delivery Date</code> sont par ailleurs interprétées comme des chaînes de caractères et devront faire l'objet d'une vérification avant leur utilisation comme variables temporelles.

</div>

### 1.3. Analyse des valeurs manquantes

Cette étape vise à identifier et quantifier les valeurs manquantes dans la table `Sales`. Leur analyse permet de déterminer si certaines variables sont incomplètes et d'évaluer si ces absences constituent une anomalie de qualité ou peuvent s'expliquer par une logique métier.

In [6]:
# Comptage des valeurs manquantes par colonne
sales.isna().sum()

Order Number         0
Line Item            0
Order Date           0
Delivery Date    49719
CustomerKey          0
StoreKey             0
ProductKey           0
Quantity             0
Currency Code        0
dtype: int64

In [7]:
# Calcul du pourcentage de valeurs manquantes par colonne
(sales.isna().mean() * 100).round(2)

Order Number      0.00
Line Item         0.00
Order Date        0.00
Delivery Date    79.06
CustomerKey       0.00
StoreKey          0.00
ProductKey        0.00
Quantity          0.00
Currency Code     0.00
dtype: float64

<div style=" display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;
">

<strong>Observation :</strong> La colonne <code>Delivery Date</code> est la seule variable contenant des valeurs manquantes, avec 49 719 valeurs absentes, soit 79,06 % des lignes de commande. Toutes les autres colonnes sont complètes. Ce taux élevé nécessite une investigation afin de déterminer si ces absences correspondent à une logique métier ou à un problème de qualité des données.

</div>

#### 1.3.1 Investigation des valeurs manquantes de `Delivery Date`

Le taux élevé de valeurs manquantes dans `Delivery Date` nécessite d'en rechercher l'origine. Nous vérifions dans un premier temps si ces absences sont associées à certains magasins (`StoreKey`), ce qui pourrait révéler une logique liée au mode de vente.

In [8]:
# Identification des StoreKey associés aux dates de livraison manquantes
sales.loc[sales["Delivery Date"].isna(), "StoreKey"].value_counts().sort_index()

StoreKey
1      292
2       18
4      431
5      892
6      615
8     1360
9     1577
10    1269
12     224
13     184
14     126
15     235
16     214
17     210
18     194
19     635
20     443
21     391
22     800
23     674
24     703
26     295
27     721
28     214
29    1011
30     976
31     417
32     454
33     498
34     448
36    1049
37     995
38    1034
39    1054
40    1050
41     444
42    1017
43    1340
44    1436
45    1471
47    1348
48    1289
49    1041
50    1519
51    1356
53    1284
54    1498
55    1518
56    1287
57    1442
59    1472
61    1485
62     778
63     948
64    1353
65    1395
66    1295
Name: count, dtype: int64

In [9]:
# Comparaison de la présence d'une date de livraison selon le StoreKey
pd.crosstab(
    sales["StoreKey"].eq(0),
    sales["Delivery Date"].isna(),
    rownames=["StoreKey = 0"],
    colnames=["Delivery Date manquante"]
)

Delivery Date manquante,False,True
StoreKey = 0,,
False,0,49719
True,13165,0


#### 1.3.2 Vérification de la signification de `StoreKey = 0`

La relation observée montre que toutes les lignes associées à `StoreKey = 0` possèdent une date de livraison, tandis que celles associées à un autre magasin n'en possèdent aucune. Nous consultons ponctuellement la table `Stores` afin de vérifier si `StoreKey = 0` correspond à un mode de vente particulier.

In [10]:
# Chargement ponctuel de la table Stores pour vérifier la signification de StoreKey = 0
stores = pd.read_csv("../data/raw/Stores.csv")

# Recherche des informations associées à StoreKey = 0
stores.loc[stores["StoreKey"] == 0]

,StoreKey,Country,State,Square Meters,Open Date
66,0,Online,Online,NaN,1/1/2010


<div style=" display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #F5BB27; background-color: #FFF8D6; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;
">

<strong>Conclusion de l'investigation :</strong> L'analyse croisée montre que les 49 719 valeurs manquantes de <code>Delivery Date</code> concernent exclusivement les ventes associées à un <code>StoreKey</code> différent de 0, tandis que les 13 165 lignes associées à <code>StoreKey = 0</code> disposent toutes d'une date de livraison. La table <code>Stores</code> confirme que <code>StoreKey = 0</code> correspond au canal de vente <strong>Online</strong>. L'absence de <code>Delivery Date</code> pour les autres magasins est donc cohérente avec la logique métier des ventes réalisées en magasin physique et ne constitue pas une anomalie de qualité des données.

</div>


<div style=" display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #0ba84f; background-color: #ddffe7; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;
">

<strong>Décision :</strong> Les valeurs manquantes de <code>Delivery Date</code> sont conservées, car elles correspondent à une absence de livraison attendue pour les ventes réalisées en magasin physique.

</div>

### 1.4. Vérification des doublons et de l'unicité des clés

Cette étape a comme objectif de détecter d'éventuelles lignes dupliquées et à vérifier l'unicité des identifiants de la table `Sales`. Nous cherchons notamment à confirmer si la combinaison `Order Number` et `Line Item` permet d'identifier de manière unique chaque ligne de commande.

In [11]:
# Vérification des lignes entièrement dupliquées (Doublon exact)
print(f'Le nombre des lignes entièrement dupliquées dans la table "Sales" est : {sales.duplicated().sum()}')

Le nombre des lignes entièrement dupliquées dans la table "Sales" est : 0


In [12]:
# Vérification de l'unicité des numéros de commande (ID métier de commande)
print(f"Nombre de lignes de commande : {len(sales)}")
print(f"Nombre de commandes distinctes : {sales['Order Number'].nunique()}")

Nombre de lignes de commande : 62884
Nombre de commandes distinctes : 26326


In [13]:
# Vérification de l'unicité de la clé candidate composée (Order Number, Line Item)
duplicate_keys = sales.duplicated(subset=["Order Number", "Line Item"]).sum()
print(f"Nombre de doublons sur la clé (Order Number, Line Item) : {duplicate_keys}")

Nombre de doublons sur la clé (Order Number, Line Item) : 0


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;">

<strong>Observation :</strong> La table <code>Sales</code> ne contient aucune ligne entièrement dupliquée. Les 62 884 lignes de commande correspondent à 26 326 commandes distinctes, confirmant qu'un même <code>Order Number</code> peut être associé à plusieurs lignes de produits. En revanche, aucune duplication n'est observée sur la combinaison <code>(Order Number, Line Item)</code>, ce qui confirme son unicité et permet de la considérer comme une clé candidate composée pour identifier chaque ligne de commande.

</div>

### 1.5. Vérification de la cohérence des valeurs

Au-delà de la structure des données, il est nécessaire de s'assurer que les valeurs enregistrées restent cohérentes avec leur signification métier. Nous recherchons ici d'éventuelles valeurs impossibles, inattendues ou hors domaine susceptibles d'affecter la fiabilité des analyses.

In [14]:
# Vérification des valeurs présentes dans la colonne Quantity
sales["Quantity"].value_counts().sort_index()

Quantity
1     18311
2     13611
3     10907
4      5441
5      3619
6      3699
7      3639
8      1849
9       895
10      913
Name: count, dtype: int64

In [15]:
# Vérification des valeurs présentes dans la colonne Currency Code
sales["Currency Code"].value_counts().sort_index()

Currency Code
AUD     2941
CAD     5415
EUR    12621
GBP     8140
USD    33767
Name: count, dtype: int64

In [16]:
# Chargement ponctuel de la table Exchange_Rates pour vérifier les devises disponibles
exchange_rates = pd.read_csv("../data/raw/Exchange_Rates.csv")

# Identification des devises disponibles dans la table Exchange_Rates
exchange_rates["Currency"].unique()

<StringArray>
['USD', 'CAD', 'AUD', 'EUR', 'GBP']
Length: 5, dtype: str

<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;">

<strong>Observation :</strong> Les quantités vendues sont toutes comprises entre 1 et 10, sans valeur nulle ou négative. La colonne <code>Currency Code</code> contient cinq devises distinctes (<code>AUD</code>, <code>CAD</code>, <code>EUR</code>, <code>GBP</code> et <code>USD</code>), toutes présentes dans la table de référence <code>Exchange_Rates</code>. Aucune valeur incohérente n'a été identifiée sur les domaines contrôlés.

</div>

### 1.6. Analyse et cohérence des données temporelles

Les dates jouent un rôle central dans l'analyse des ventes, notamment pour étudier leur évolution, la saisonnalité ou les délais de livraison. Avant toute analyse temporelle, nous devons vérifier leur format, leur couverture chronologique et la cohérence entre les dates de commande et de livraison.

In [17]:
# Conversion des colonnes de dates au format datetime
order_date = pd.to_datetime(sales["Order Date"], format="%m/%d/%Y", errors="coerce")
delivery_date = pd.to_datetime(sales["Delivery Date"], format="%m/%d/%Y", errors="coerce")

# Vérification des dates de commande invalides après conversion
print(f"Nombre de dates de commande invalides : {order_date.isna().sum()}")

Nombre de dates de commande invalides : 0


In [18]:
# Vérification des dates de livraison invalides après conversion
missing_delivery_before = sales["Delivery Date"].isna().sum()
missing_delivery_after = delivery_date.isna().sum()

print(f"Valeurs manquantes avant conversion : {missing_delivery_before}")
print(f"Valeurs manquantes après conversion : {missing_delivery_after}")
print(f"Dates de livraison invalides : {missing_delivery_after - missing_delivery_before}")

Valeurs manquantes avant conversion : 49719
Valeurs manquantes après conversion : 49719
Dates de livraison invalides : 0


In [19]:
# Identification de la période couverte par les commandes
print(f"Première date de commande : {order_date.min().date()}")
print(f"Dernière date de commande : {order_date.max().date()}")

Première date de commande : 2016-01-01
Dernière date de commande : 2021-02-20


In [20]:
# Vérification de la répartition des lignes de commande par année
order_date.dt.year.value_counts().sort_index()

Order Date
2016     6905
2017     7942
2018    14188
2019    21611
2020    11026
2021     1212
Name: count, dtype: int64

In [21]:
# Vérification des livraisons antérieures à la date de commande
invalid_delivery_order = (delivery_date < order_date).sum()
print(f"Nombre de livraisons antérieures à la date de commande : {invalid_delivery_order}")

Nombre de livraisons antérieures à la date de commande : 0


In [22]:
# Calcul et synthèse des délais de livraison en jours
delivery_delay = (delivery_date - order_date).dt.days

print(f"Nombre de lignes livrées : {delivery_delay.count():,.0f}".replace(",", " "))
print(f"Délai moyen : {delivery_delay.mean():.2f}".replace(".", ",") + " jours")
print(f"Délai médian : {delivery_delay.median():.0f} jours")
print(f"1er quartile : {delivery_delay.quantile(0.25):.0f} jours")
print(f"3e quartile : {delivery_delay.quantile(0.75):.0f} jours")
print(f"Délai minimum : {delivery_delay.min():.0f} jour")
print(f"Délai maximum : {delivery_delay.max():.0f} jours")

Nombre de lignes livrées : 13 165
Délai moyen : 4,53 jours
Délai médian : 4 jours
1er quartile : 3 jours
3e quartile : 6 jours
Délai minimum : 1 jour
Délai maximum : 17 jours


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;">

<strong>Observation :</strong> Les dates de commande et les dates de livraison renseignées sont toutes conformes au format attendu, sans valeur invalide détectée. Les commandes couvrent la période du <strong>1er janvier 2016 au 20 février 2021</strong>, avec des observations présentes chaque année ; l'année 2021 est toutefois partielle. Aucune livraison n'est antérieure à sa date de commande. Pour les 13 165 lignes disposant d'une date de livraison, les délais sont compris entre 1 et 17 jours, avec une médiane de 4 jours et une moyenne d'environ 4,5 jours, sans incohérence temporelle manifeste.

</div>

### 1.7. Vérification de l'intégrité référentielle

La table `Sales` contient plusieurs clés permettant de relier chaque ligne de commande aux entités de référence du dataset. Nous vérifions ici que les identifiants présents dans `Sales` possèdent bien une correspondance dans les tables associées, afin de détecter d'éventuelles références orphelines.

In [23]:
# Chargement ponctuel de la table Customers pour vérifier les références clients
customers = pd.read_csv("../data/raw/Customers.csv", encoding="cp1252")

In [24]:
# Identification des CustomerKey de Sales absents de la table Customers
orphan_customer_keys = ~sales["CustomerKey"].isin(customers["CustomerKey"])
print(f"Nombre de lignes de commande avec un CustomerKey sans correspondance : {orphan_customer_keys.sum()}")

Nombre de lignes de commande avec un CustomerKey sans correspondance : 0


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> Tous les <code>CustomerKey</code> présents dans <code>Sales</code> possèdent une correspondance dans la table <code>Customers</code> ; aucune référence client orpheline n'a été détectée.</div>

In [25]:
# Chargement ponctuel de la table Products pour vérifier les références produits
products = pd.read_csv("../data/raw/Products.csv")

In [26]:
# Identification des ProductKey de Sales absents de la table Products
orphan_product_keys = ~sales["ProductKey"].isin(products["ProductKey"])
print(f"Nombre de lignes de commande avec un ProductKey sans correspondance : {orphan_product_keys.sum()}")

Nombre de lignes de commande avec un ProductKey sans correspondance : 0


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> Aucun <code>ProductKey</code> orphelin n'a été identifié : chaque produit référencé dans <code>Sales</code> existe bien dans la table <code>Products</code>.</div>

In [27]:
# La table Stores a été chargée précédemment dans le notebook
# Identification des StoreKey de Sales absents de la table Stores

orphan_store_keys = ~sales["StoreKey"].isin(stores["StoreKey"])
print(f"Nombre de lignes de commande avec un StoreKey sans correspondance : {orphan_store_keys.sum()}")

Nombre de lignes de commande avec un StoreKey sans correspondance : 0


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> L'ensemble des <code>StoreKey</code> utilisés dans <code>Sales</code> est référencé dans la table <code>Stores</code>, y compris la valeur <code>0</code> associée au canal de vente Online ; aucune référence magasin orpheline n'est donc présente.</div>

In [28]:
# La table Exchange_Rates a été chargée précédemment dans le notebook
# Conversion temporaire de sa colonne Date au format datetime

exchange_date = pd.to_datetime(
    exchange_rates["Date"],
    format="%m/%d/%Y",
    errors="coerce"
)

print(f"Nombre de dates invalides dans Exchange_Rates : {exchange_date.isna().sum()}")

Nombre de dates invalides dans Exchange_Rates : 0


In [29]:
# Vérification de la correspondance entre Sales et Exchange_Rates sur la date et la devise

sales_exchange_check = pd.DataFrame({
    "Order Date": order_date,
    "Currency Code": sales["Currency Code"]
})

exchange_check = pd.DataFrame({
    "Date": exchange_date,
    "Currency": exchange_rates["Currency"]
})

sales_exchange_check = sales_exchange_check.merge(
    exchange_check,
    how="left",
    left_on=["Order Date", "Currency Code"],
    right_on=["Date", "Currency"],
    indicator=True
)

orphan_exchange_rates = (sales_exchange_check["_merge"] == "left_only").sum()

print(f"Nombre de lignes de commande sans taux de change correspondant : {orphan_exchange_rates}")

Nombre de lignes de commande sans taux de change correspondant : 0


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> Chaque combinaison <code>(Order Date, Currency Code)</code> utilisée dans <code>Sales</code> trouve une correspondance sur <code>(Date, Currency)</code> dans <code>Exchange_Rates</code> ; aucune ligne de commande ne présente donc de référence orpheline vis-à-vis des taux de change.</div>

<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #F5BB27; background-color: #FFF8D6; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Conclusion de l'investigation :</strong> Les contrôles d'intégrité référentielle ne révèlent aucune référence orpheline dans <code>Sales</code> : les relations avec <code>Customers</code>, <code>Products</code>, <code>Stores</code> et <code>Exchange_Rates</code> sont toutes cohérentes avec le modèle de données attendu.</div>

### 1.8. Vérification de la cohérence de Line Item

Après avoir confirmé l'unicité de la combinaison `(Order Number, Line Item)`, nous examinons la numérotation des lignes au sein de chaque commande. Ce contrôle exploratoire repose sur l'hypothèse que les `Line Item` suivent une séquence continue à partir de 1 ; cette règle n'étant pas explicitement documentée dans le dictionnaire de données, les éventuelles ruptures détectées devront être investiguées avant d'être considérées comme des anomalies.

In [30]:
# Vérification des valeurs présentes dans la colonne Line Item
sales["Line Item"].value_counts().sort_index()

Line Item
1    26326
2    16789
3    10248
4     5125
5     2175
6     1476
7      745
Name: count, dtype: int64

In [31]:
# Vérification de la continuité des Line Item au sein de chaque commande

line_item_check = sales.groupby("Order Number")["Line Item"].agg(["count", "max"])

invalid_line_item_sequences = (
    line_item_check["count"] != line_item_check["max"]
).sum()

print(f"Nombre de commandes avec une séquence de Line Item incohérente : {invalid_line_item_sequences}")

Nombre de commandes avec une séquence de Line Item incohérente : 621


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> Le contrôle exploratoire de la numérotation des <code>Line Item</code> identifie <strong>621 commandes</strong> pour lesquelles le nombre de lignes diffère du numéro de <code>Line Item</code> maximal, suggérant une possible rupture dans la séquence attendue <code>1...n</code>. Ce résultat nécessite une investigation avant de pouvoir être interprété comme une anomalie.</div>

#### 1.8.1. Investigation des ruptures de séquence de `Line Item`

Le contrôle exploratoire a identifié 621 commandes pour lesquelles le nombre de lignes diffère du numéro de `Line Item` maximal. Nous examinons ces commandes afin de comprendre l'origine de ces écarts et de déterminer s'ils correspondent à de véritables anomalies ou à une particularité de la structure des données.

In [32]:
# Synthèse des premières commandes présentant une rupture de séquence
line_item_investigation = (
    sales.groupby("Order Number")["Line Item"]
    .agg(
        nombre_lignes="count",
        line_item_max="max",
        line_items=lambda x: sorted(x.tolist())
    )
)

line_item_investigation = line_item_investigation[
    line_item_investigation["nombre_lignes"] != line_item_investigation["line_item_max"]
]

line_item_investigation.head(10)

,nombre_lignes,line_item_max,line_items
Order Number,,,
372008,2,3,"[1, 3]"
374000,5,6,"[1, 2, 3, 5, 6]"
384000,3,4,"[1, 2, 4]"
385009,2,3,"[1, 3]"
390001,5,6,"[1, 2, 3, 4, 6]"
398003,3,4,"[1, 3, 4]"
409011,5,6,"[1, 2, 3, 4, 6]"
411006,3,4,"[1, 2, 4]"
420005,5,6,"[1, 2, 4, 5, 6]"


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> Sur les 10 premiers cas, j'observe quelque chose : il semble manquer exactement un numéro à chaque fois. Vérifions si ceci est le cas pour toutes les 621 commandes ou bien il y'a des exceptions.</div>

In [33]:
# Calcul du nombre de Line Item manquants pour chaque commande à investiguer

line_item_investigation["nombre_manquants"] = (
    line_item_investigation["line_item_max"]
    - line_item_investigation["nombre_lignes"]
)

line_item_investigation["nombre_manquants"].value_counts().sort_index()

nombre_manquants
1    612
2      8
3      1
Name: count, dtype: int64

<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> Les 612 cas avec un seul trou semblent constituer le comportement dominant avec 612 commandes. Les 9 autres peuvent nous aider à comprendre si on observe simplement le même phénomène plusieurs fois ou quelque chose de différent. Vérifions les 9 cas le splus atypiques càd ceux où il manque 2 ou 3 numéros.</div>

In [34]:
# Affichage des commandes présentant plusieurs Line Item manquants

line_item_investigation[
    line_item_investigation["nombre_manquants"] > 1
]

,nombre_lignes,line_item_max,line_items,nombre_manquants
Order Number,,,,
783003,4,6,"[1, 2, 4, 6]",2
906002,3,5,"[1, 3, 5]",2
909001,4,6,"[1, 3, 5, 6]",2
1014001,5,7,"[1, 2, 3, 5, 7]",2
1233014,5,7,"[1, 2, 4, 5, 7]",2
1506001,4,6,"[1, 3, 5, 6]",2
1508054,4,7,"[1, 3, 5, 7]",3
1807023,5,7,"[1, 2, 3, 5, 7]",2
1851013,5,7,"[1, 2, 4, 5, 7]",2


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #F5BB27; background-color: #FFF8D6; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Conclusion de l'investigation :</strong> Les 621 commandes identifiées présentent bien des ruptures dans la numérotation des <code>Line Item</code>. Parmi elles, 612 comportent un seul numéro manquant, 8 en comportent deux et 1 en comporte trois. L'examen des cas les plus atypiques confirme qu'il s'agit du même phénomène de discontinuité de numérotation. Toutefois, le dictionnaire de données indique uniquement que <code>Line Item</code> identifie les produits au sein d'une commande et ne précise pas que leur numérotation doit obligatoirement former une séquence continue. Ces ruptures ne peuvent donc pas être considérées comme des anomalies de qualité sur la seule base de ce contrôle.</div>

<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #0ba84f; background-color: #ddffe7; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Décision :</strong> Les valeurs de <code>Line Item</code> sont conservées telles quelles. En l'absence de règle métier imposant une numérotation continue, aucune correction ne sera appliquée. La combinaison <code>(Order Number, Line Item)</code> reste néanmoins valide comme clé candidate puisqu'elle est unique pour l'ensemble des lignes de la table <code>Sales</code>.</div>

### Bilan de l'audit de la table `Sales`

La table `Sales` contient **62 884 lignes de commande réparties sur 26 326 commandes et 9 colonnes**. Son audit a permis d'évaluer la qualité et la cohérence des données transactionnelles, notamment à travers l'analyse des valeurs manquantes, de l'unicité des lignes de commande, des valeurs métier, des données temporelles et de l'intégrité référentielle.

| Sous-étape | Ce que nous avons vérifié | Résultat |
|---|---|---|
| **1.1 Chargement et première inspection** | Chargement du CSV et aperçu des premières lignes | Fichier correctement chargé |
| **1.2 Structure et dimensions** | Nombre de lignes/colonnes, colonnes et types | **62 884 lignes × 9 colonnes** ; `Order Date` et `Delivery Date` initialement au format texte |
| **1.3 Valeurs manquantes** | Nombre et proportion de valeurs manquantes + investigation de `Delivery Date` | **49 719 `Delivery Date` manquantes (79,06 %)**, exclusivement associées aux ventes physiques ; aucune correction nécessaire |
| **1.4 Doublons et unicité** | Doublons complets, unicité de `Order Number` et de la combinaison (`Order Number`, `Line Item`) | **0 doublon complet** ; **26 326 commandes** ; combinaison (`Order Number`, `Line Item`) unique et retenue comme clé candidate |
| **1.5 Cohérence des valeurs** | Domaine de `Quantity` et validité de `Currency Code` | `Quantity` comprise entre **1 et 10** ; **5 devises** présentes et toutes référencées dans `Exchange_Rates` |
| **1.6 Données temporelles** | Validité des dates, période couverte, cohérence commande/livraison et délais | **0 date invalide** ; données du **01/01/2016 au 20/02/2021** ; aucune livraison antérieure à la commande ; délais de livraison cohérents |
| **1.7 Intégrité référentielle** | Références vers `Customers`, `Products`, `Stores` et `Exchange_Rates` | **0 référence orpheline** détectée |
| **1.8 Cohérence de `Line Item`** | Continuité de la numérotation des lignes au sein des commandes + investigation des ruptures | **621 commandes** présentent des ruptures de séquence ; aucune règle métier n'imposant une numérotation continue, les valeurs sont conservées sans modification |

## 2. Audit de la table `Customers`

Après avoir validé la qualité et la cohérence de la table transactionnelle centrale `Sales`, nous poursuivons l'audit avec la table `Customers`. Cette table contient les informations descriptives et démographiques des clients et constitue l'un des principaux axes d'analyse des ventes, notamment pour étudier les comportements et les profils de la clientèle.

### 2.1. Chargement et 1ère inspection des données

Nous commençons par charger la table `Customers` et afficher un aperçu de ses premières lignes afin d'identifier les variables disponibles, d'observer leur organisation et de nous familiariser avec les informations décrivant les clients.

In [35]:
# Chargement des données clients dans un DataFrame
customers = pd.read_csv("../data/raw/Customers.csv", encoding="cp1252")

# Aperçu des premières lignes de la table Customers
customers.head()

,CustomerKey,Gender,Name,City,State Code,State,Zip Code,Country,Continent,Birthday
0,301,Female,Lilly Harding,WANDEARAH EAST,SA,South Australia,5523,Australia,Australia,7/3/1939
1,325,Female,Madison Hull,MOUNT BUDD,WA,Western Australia,6522,Australia,Australia,9/27/1979
2,554,Female,Claire Ferres,WINJALLOK,VIC,Victoria,3380,Australia,Australia,5/26/1947
3,786,Male,Jai Poltpalingada,MIDDLE RIVER,SA,South Australia,5223,Australia,Australia,9/17/1957
4,1042,Male,Aidan Pankhurst,TAWONGA SOUTH,VIC,Victoria,3698,Australia,Australia,11/19/1965


### 2.2. Vérification de la structure et des dimensions

Après ce premier aperçu, nous examinons les dimensions de la table ainsi que les types de données détectés par Pandas. Ce contrôle permet de connaître le volume de données à auditer et d'identifier d'éventuels types de variables qui devront être adaptés pour les analyses futures.

In [36]:
# Vérification des dimensions de la table Customers
print(f"Nombre de lignes : {customers.shape[0]}")
print(f"Nombre de colonnes : {customers.shape[1]}")

Nombre de lignes : 15266
Nombre de colonnes : 10


In [37]:
# Vérification de la structure et des types de données
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 15266 entries, 0 to 15265
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   CustomerKey  15266 non-null  int64
 1   Gender       15266 non-null  str  
 2   Name         15266 non-null  str  
 3   City         15266 non-null  str  
 4   State Code   15256 non-null  str  
 5   State        15266 non-null  str  
 6   Zip Code     15266 non-null  str  
 7   Country      15266 non-null  str  
 8   Continent    15266 non-null  str  
 9   Birthday     15266 non-null  str  
dtypes: int64(1), str(9)
memory usage: 1.2 MB


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> La table <code>Customers</code> contient 15 266 lignes et 10 colonnes. La majorité des variables sont complètes, à l'exception de <code>State Code</code> qui compte 10 valeurs manquantes. La colonne <code>Birthday</code> est actuellement interprétée comme une chaîne de caractères et devra faire l'objet d'un contrôle spécifique avant son utilisation dans les analyses temporelles.</div>

### 2.3. Analyse des valeurs manquantes

Nous analysons maintenant la présence de valeurs manquantes dans la table `Customers` afin d'identifier les variables concernées et de mesurer l'ampleur des données absentes. Les éventuels cas détectés seront ensuite examinés pour déterminer s'ils traduisent un problème de qualité ou une particularité des données.

In [38]:
# Synthèse des valeurs manquantes par colonne
missing_values = pd.DataFrame({
    "nombre_manquant": customers.isna().sum(),
    "pourcentage_manquant": (customers.isna().mean() * 100).round(2)
})

missing_values

,nombre_manquant,pourcentage_manquant
CustomerKey,0,0.00
Gender,0,0.00
Name,0,0.00
City,0,0.00
State Code,10,0.07
State,0,0.00
Zip Code,0,0.00
Country,0,0.00
Continent,0,0.00
Birthday,0,0.00


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> La colonne <code>State Code</code> est la seule variable contenant des valeurs manquantes, avec 10 valeurs absentes sur 15 266 lignes, soit 0,07 % des données. Toutes les autres colonnes sont complètes. Ces quelques valeurs manquantes nécessitent une investigation afin d'en comprendre l'origine et de déterminer si un traitement est nécessaire.</div>

#### 2.3.1. Investigation des valeurs manquantes de `State Code`

Les 10 valeurs manquantes de `State Code` représentent une part très faible de la table, mais leur origine doit être comprise avant de décider d'un éventuel traitement. Nous examinons donc les informations géographiques associées aux clients concernés afin de rechercher un éventuel comportement commun.

In [39]:
# Affichage des informations géographiques des clients sans State Code
customers.loc[
    customers["State Code"].isna(),
    ["CustomerKey", "City", "State Code", "State", "Zip Code", "Country", "Continent"]
]

,CustomerKey,City,State Code,State,Zip Code,Country,Continent
5304,729681,Polvica,NaN,Napoli,80035,Italy,Europe
5316,732289,Varcaturo,NaN,Napoli,80014,Italy,Europe
5372,742042,Casaferro,NaN,Napoli,80034,Italy,Europe
5377,742886,Terzigno,NaN,Napoli,80040,Italy,Europe
5378,743343,Pomigliano D'Arco,NaN,Napoli,80038,Italy,Europe
5485,759705,Casilli,NaN,Napoli,80047,Italy,Europe
5525,765589,Pompei Scavi,NaN,Napoli,80045,Italy,Europe
5531,766410,Licola,NaN,Napoli,80078,Italy,Europe
5631,781667,Napoli,NaN,Napoli,80134,Italy,Europe
5695,789177,Mariglianella,NaN,Napoli,80030,Italy,Europe


In [40]:
# Vérification des State Code pour l'ensemble des clients de Napoli
customers.loc[
    customers["State"] == "Napoli",
    ["State", "State Code"]
].value_counts(dropna=False)

State   State Code
Napoli  NaN           10
Name: count, dtype: int64

In [41]:
# Vérification du couple State / State Code pour Napoli
customers.loc[
    (customers["Country"] == "Italy") & (customers["State"] == "Napoli"),
    ["State", "State Code"]
].drop_duplicates()

,State,State Code
5304,Napoli,NaN


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #F5BB27; background-color: #FFF8D6; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Conclusion de l'investigation :</strong> Les 10 valeurs manquantes de <code>State Code</code> concernent exclusivement les clients rattachés à <code>Napoli</code>, en Italie. Tous les clients de cette région présents dans le dataset sont concernés, tandis que leur <code>State</code>, leur ville, leur code postal et leur pays restent renseignés. L'absence de <code>State Code</code> apparaît donc comme un comportement systématique associé à <code>Napoli</code> dans la source plutôt que comme des valeurs manquantes dispersées aléatoirement.</div>

<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #0ba84f; background-color: #ddffe7; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Décision :</strong> Une vérification auprès d'une source administrative officielle italienne confirme que le code associé à <code>Napoli</code> est <code>NA</code>, selon le même système de codification que celui observé pour les autres provinces italiennes du dataset. Donc, les 10 valeurs manquantes de <code>State Code</code> associées à <code>Napoli</code> seront remplacées par <code>NA</code>. Cette correction permet de compléter l'information géographique sans recourir à une estimation.</div>

### 2.4. Doublons et unicité de la clé `CustomerKey`

Nous vérifions maintenant l'absence de lignes entièrement dupliquées ainsi que l'unicité de `CustomerKey`, identifié comme clé primaire de la table `Customers` dans le dictionnaire de données. Cette vérification permet de confirmer que chaque identifiant correspond à une seule ligne client et que la table ne contient pas d'enregistrements dupliqués.

In [42]:
# Recherche des lignes entièrement dupliquées dans Customers
customers.duplicated().sum()

np.int64(0)

In [43]:
# Vérification de l'unicité de CustomerKey
print(f"Nombre de lignes : {len(customers)}")
print(f"Nombre de CustomerKey uniques : {customers['CustomerKey'].nunique()}")
print(f"Nombre de CustomerKey dupliqués : {customers['CustomerKey'].duplicated().sum()}")

Nombre de lignes : 15266
Nombre de CustomerKey uniques : 15266
Nombre de CustomerKey dupliqués : 0


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> Aucun doublon complet n'est présent dans la table <code>Customers</code>. Les 15 266 valeurs de <code>CustomerKey</code> sont également toutes distinctes, ce qui confirme l'unicité de cette colonne et sa cohérence avec son rôle de clé primaire.</div>

### 2.5. Vérification de la cohérence des variables catégorielles et géographiques

Dans cette partie, nous examinons les principales variables catégorielles et géographiques de la table `Customers` afin d'identifier d'éventuelles modalités inattendues ou incohérences. L'analyse portera notamment sur `Gender`, `Country` et `Continent`, ainsi que sur la cohérence des associations géographiques entre ces variables.

#### 2.5.1. Vérification de la variable `Gender`

Nous commençons par examiner les différentes modalités présentes dans `Gender` ainsi que leur fréquence. Ce contrôle permet notamment de détecter d'éventuelles valeurs inattendues, variantes d'écriture ou catégories anormalement représentées.

In [44]:
# Analyse des modalités de la variable Gender
customers["Gender"].value_counts(dropna=False)

Gender
Male      7748
Female    7518
Name: count, dtype: int64

<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> La variable <code>Gender</code> contient uniquement les deux modalités <code>Male</code> et <code>Female</code>, avec respectivement 7 748 et 7 518 clients. Aucune valeur manquante, modalité inattendue ou variation d'écriture n'est détectée.</div>

#### 2.5.2. Vérification des variables `Country` et `Continent`

Nous poursuivons l'analyse avec les variables `Country` et `Continent` afin d'identifier les zones géographiques représentées dans le dataset et de vérifier la cohérence de leur association. Chaque pays devrait notamment être rattaché à un seul continent dans l'ensemble de la table.

In [45]:
# Identification des associations distinctes entre Country et Continent
customers[
    ["Country", "Continent"]
].drop_duplicates().sort_values(["Continent", "Country"])

,Country,Continent
0,Australia,Australia
4446,France,Europe
2973,Germany,Europe
5116,Italy,Europe
5761,Netherlands,Europe
6494,United Kingdom,Europe
1420,Canada,North America
8438,United States,North America


In [46]:
# Vérification du nombre de continents associés à chaque pays
country_continent_check = (
    customers.groupby("Country")["Continent"]
    .nunique()
)

country_continent_check

Country
Australia         1
Canada            1
France            1
Germany           1
Italy             1
Netherlands       1
United Kingdom    1
United States     1
Name: Continent, dtype: int64

<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> La table <code>Customers</code> contient 8 pays répartis sur 3 continents. Chaque pays est associé à un unique continent et les correspondances observées sont géographiquement cohérentes. Aucune incohérence n'est détectée entre les variables <code>Country</code> et <code>Continent</code>.</div>

#### 2.5.3. Vérification de la cohérence entre `State` et `State Code`

Nous examinons maintenant la relation entre `State` et `State Code` afin de vérifier la cohérence de leur association dans le dataset. L'objectif est notamment de détecter les cas où un même État, province ou région serait associé à plusieurs codes différents.

In [47]:
# Vérification de la cohérence entre State et State Code
state_code_check = (
    customers.groupby(["Country", "State"])["State Code"]
    .nunique(dropna=False)
)

inconsistent_state_codes = state_code_check[state_code_check > 1]

print(f"Nombre d'associations incohérentes : {len(inconsistent_state_codes)}")
inconsistent_state_codes

Nombre d'associations incohérentes : 0


Series([], Name: State Code, dtype: int64)

<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> Aucune association incohérente n'est détectée entre <code>State</code> et <code>State Code</code> : chaque combinaison <code>Country + State</code> est associée à un seul code dans le dataset. Le cas particulier de <code>Napoli</code>, dont le code est manquant, a été identifié et traité séparément lors de l'analyse des valeurs manquantes.</div>

### 2.6. Analyse et cohérence de `Birthday`

La variable `Birthday` est actuellement stockée sous forme de chaîne de caractères. Nous vérifions d'abord que l'ensemble des valeurs respecte un format de date valide avant d'examiner leur cohérence temporelle et leur pertinence pour les futures analyses liées à l'âge des clients.

In [48]:
# Conversion temporaire de Birthday au format datetime pour contrôler sa validité
birthday_date = pd.to_datetime(
    customers["Birthday"],
    format="%m/%d/%Y",
    errors="coerce"
)

print(f"Nombre de dates invalides : {birthday_date.isna().sum()}")

Nombre de dates invalides : 0


In [49]:
# Vérification des bornes des dates de naissance
print(f"Date de naissance la plus ancienne : {birthday_date.min().date()}")
print(f"Date de naissance la plus récente : {birthday_date.max().date()}")

Date de naissance la plus ancienne : 1935-02-03
Date de naissance la plus récente : 2002-02-18


In [50]:
# Calcul de l'âge des clients au moment de leurs commandes

customer_birthdays = customers[["CustomerKey"]].copy()
customer_birthdays["Birthday"] = birthday_date

customer_age_check = sales[["CustomerKey", "Order Date"]].merge(
    customer_birthdays,
    on="CustomerKey",
    how="left"
)

customer_age_check["Order Date"] = pd.to_datetime(
    customer_age_check["Order Date"],
    format="%m/%d/%Y"
)

customer_age_check["Age"] = (
    customer_age_check["Order Date"].dt.year
    - customer_age_check["Birthday"].dt.year
    - (
        customer_age_check["Order Date"].dt.strftime("%m%d")
        < customer_age_check["Birthday"].dt.strftime("%m%d")
    )
)

print(f"Âge minimum au moment d'une commande : {customer_age_check['Age'].min()} ans")
print(f"Âge maximum au moment d'une commande : {customer_age_check['Age'].max()} ans")

Âge minimum au moment d'une commande : 14 ans
Âge maximum au moment d'une commande : 85 ans


#### 2.6.1. Investigation des âges les plus faibles

Le calcul de l'âge au moment des commandes fait apparaître un âge minimum de 14 ans. Cette valeur n'étant pas nécessairement erronée mais pouvant représenter un cas atypique, nous examinons les commandes associées aux clients les plus jeunes afin de déterminer s'il s'agit d'un cas isolé ou d'un comportement plus largement présent dans les données.

In [55]:
# Identification des commandes passées par des clients âgés de moins de 18 ans
underage_orders = customer_age_check[
    customer_age_check["Age"] < 18
]

ages = sorted(int(age) for age in underage_orders["Age"].unique())

print(f"Nombre de lignes de commande concernées : {len(underage_orders)}")
print(f"Nombre de clients concernés : {underage_orders['CustomerKey'].nunique()}")
print(f"Âges observés : {ages}")


Nombre de lignes de commande concernées : 1335
Nombre de clients concernés : 351
Âges observés : [14, 15, 16, 17]


In [56]:
# Répartition des clients mineurs selon leur âge au moment de la commande
underage_orders.groupby("Age")["CustomerKey"].nunique()

Age
14     20
15     76
16    130
17    223
Name: CustomerKey, dtype: int64

<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #F5BB27; background-color: #FFF8D6; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Conclusion de l'investigation :</strong> L'âge minimum observé au moment d'une commande est de <code>14 ans</code>. Au total, <code>351 clients distincts</code> ont effectué au moins une commande avant 18 ans, avec des âges compris entre <code>14 et 17 ans</code>. Les clients âgés de 17 ans sont les plus fréquemment représentés parmi ces commandes. En l'absence de règle métier imposant un âge minimum dans la documentation du dataset, ces observations ne peuvent pas être considérées comme des anomalies de données.</div>

<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #0ba84f; background-color: #ddffe7; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Décision :</strong> Les dates de naissance et les enregistrements associés aux clients mineurs sont conservés sans modification. Aucun traitement correctif ne sera appliqué en l'absence d'élément permettant d'établir qu'ils résultent d'une erreur de données.</div>

### 2.7. Vérification de la présence des clients dans `Sales`

L'intégrité référentielle de `Sales` vers `Customers` ayant déjà été validée lors de l'audit de la table transactionnelle, nous vérifions ici la relation dans le sens inverse. L'objectif est de déterminer si tous les clients enregistrés dans `Customers` apparaissent dans au moins une transaction de `Sales`, ou si la table contient également des clients sans historique d'achat.

In [57]:
# Identification des clients sans aucune transaction dans Sales
customers_without_sales = customers[
    ~customers["CustomerKey"].isin(sales["CustomerKey"])
]

print(f"Nombre total de clients : {len(customers)}")
print(f"Nombre de clients présents dans Sales : {sales['CustomerKey'].nunique()}")
print(f"Nombre de clients sans transaction : {len(customers_without_sales)}")

Nombre total de clients : 15266
Nombre de clients présents dans Sales : 11887
Nombre de clients sans transaction : 3379


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> Parmi les 15 266 clients enregistrés dans <code>Customers</code>, 11 887 apparaissent dans au moins une transaction de <code>Sales</code>, tandis que 3 379 clients, soit environ 22,1 %, ne sont associés à aucune vente sur la période couverte. Cette situation ne constitue pas une anomalie d'intégrité référentielle : la table <code>Customers</code> contient simplement des clients sans historique de transaction dans <code>Sales</code>.</div>

### Bilan de l'audit de `Customers`

La table `Customers` contient **15 266 lignes et 10 colonnes**, représentant les informations descriptives et démographiques des clients du dataset. Son audit a permis d'évaluer la qualité et la cohérence de ces données, notamment à travers l'analyse des valeurs manquantes, de l'unicité des clients, des informations géographiques, des dates de naissance et de leur présence dans l'historique des ventes.
Les principaux résultats obtenus sont synthétisés ci-dessous.

| Sous-étape | Ce que nous avons vérifié | Résultat |
|---|---|---|
| **2.1 Chargement et première inspection** | Chargement du CSV et aperçu des premières lignes | Fichier correctement chargé avec l'encodage `cp1252` |
| **2.2 Structure et dimensions** | Nombre de lignes/colonnes, colonnes et types | **15 266 lignes × 10 colonnes** ; `Birthday` initialement au format texte |
| **2.3 Valeurs manquantes** | Nombre et proportion de valeurs manquantes + investigation de `State Code` | **10 `State Code` manquants (0,07 %)**, tous associés à `Napoli` ; code officiel `NA` identifié pour correction |
| **2.4 Doublons et unicité** | Doublons complets et unicité de `CustomerKey` | **0 doublon complet** ; **15 266 `CustomerKey` uniques**, confirmant la clé primaire |
| **2.5 Cohérence catégorielle et géographique** | Modalités de `Gender`, associations `Country`–`Continent` et `State`–`State Code` | 2 modalités de `Gender` cohérentes ; **8 pays / 3 continents** ; aucune association géographique incohérente |
| **2.6 Données de naissance et âges** | Validité de `Birthday`, bornes des dates et âge au moment des commandes | **0 date invalide** ; naissances du 03/02/1935 au 18/02/2002 ; âges de **14 à 85 ans** ; présence de clients mineurs investiguée et conservée |
| **2.7 Présence dans `Sales`** | Présence des clients dans l'historique transactionnel | **11 887 clients** présents dans `Sales` ; **3 379 clients (22,1 %)** sans transaction sur la période couverte |